# Text2Cypher - 사용자 질의를 Cypher 쿼리로 변환하기

In [4]:
from dotenv import load_dotenv
from neo4j_graphrag.generation.prompts import Text2CypherTemplate
from neo4j_graphrag.llm import OpenAILLM
import re

env = load_dotenv()
print(env)

True


# 1. LLM 정의

In [5]:
llm = OpenAILLM(
    model_name="gpt-4o-mini",
    model_params={
        "temperature": 0,
    }
)

# 2. Text2Cypher 프롬프트 템플릿 확인

Neo4j GraphRAG 패키지에서 제공하는 기본 프롬프트 템플릿

In [7]:
prompt_template = Text2CypherTemplate()
print(prompt_template.template)


Task: Generate a Cypher statement for querying a Neo4j graph database from a user input.

Schema:
{schema}

Examples (optional):
{examples}

Input:
{query_text}

Do not use any properties or relationships not included in the schema.
Do not include triple backticks ``` or any additional text except the generated Cypher statement in your response.

Cypher query:



# 3. Text2Cypher 구현

1. 프롬프트 템플릿에 스키마와 질문을 삽입
2. LLM을 호출하여 Cypher 쿼리 생성
3. 생성된 쿼리를 추출하여 반환

- LLM 응답에서 Cypher 쿼리를 추출하는 함수


In [ ]:
def extract_cypher(text: str) -> str:
    """
    LLM 응답에서 Cypher 쿼리를 추출하고 포맷팅합니다.
    
    이 함수는 두 가지 작업을 수행합니다:
    1. 트리플 백틱(```) 안의 Cypher 코드를 추출
    2. 여러 단어로 된 식별자에 자동으로 백틱 추가:
        - 노드 레이블 (예: ":Data Science" → ":`Data Science`")
        - 속성 키 (예: "first name:" → "`first name`:")
        - 관계 타입 (예: "[:WORKS WITH]" → "[:`WORKS WITH`]")
        
    Args:
        text (str): Cypher 코드를 포함할 수 있는 원시 텍스트

    Returns:
        str: 올바르게 포맷팅된 Cypher 쿼리
    """
    # 트리플 백틱으로 감싸진 Cypher 코드 추출
    pattern = r"```(.*?)```"
    matches = re.findall(pattern, text, re.DOTALL)
    cypher_query = matches[0] if matches else text
    
    # 공백이 퐇마된 노드 레이블에 백틱 추가
    # 노드 레이블은 공백이 있을 경우 쿼리가 시행되지 않기 때문에 백틱(`) 으로 감싸주는 규칙을 작성해줌.
    cypher_query = re.sub(
        r":\s*(?!`\s*)(\s*)([a-zA-Z0-9_]+(?:\s+[a-zA-Z0-9_]+)+)(?!\s*`)(\s*)",
        r":`\2`",
        cypher_query,
    )
    
    # 공백이 포함된 속성 키에 백틱 추가
    cypher_query = re.sub(
        r"([,{]\s*)(?!`)([a-zA-Z0-9_]+(?:\s+[a-zA-Z0-9_]+)+)(?!`)(\s*:)",
        r"\1`\2`\3",
        cypher_query,
    )
    
    # 공백이 포함된 관계 타입에 백틱 추가
    cypher_query = re.sub(
        r"(\[\s*[a-zA-Z0-9_]*\s*:\s*)(?!`)([a-zA-Z0-9_]+(?:\s+[a-zA-Z0-9_]+)+)(?!`)(\s*(?:\]|-))",
        r"\1`\2`\3",
        cypher_query,
    )

    return cypher_query.strip()

In [10]:
def text_to_cypher(
    question: str,
    schema: str,
    examples: str = "",
) -> dict:
    """
    자연어 질문을 Cypher 쿼리로 변환합니다.
    
    Args:
        question (str): 사용자의 자연어 질문
        schema (str): Neo4j 그래프 스키마 정보
        examples (str): 예시 질문-쿼리 쌍 (선택사항)
        
    Returns:
        dict: {
            'question': 원본 질문,
            'prompt': LLM에 전달된 프롬프트,
            'llm_response': LLM의 원시 응답,
            'cypher_query': 추출된 Cypher 쿼리
        }
    """
    # 프롬프트 템플릿 생성
    template = Text2CypherTemplate()
    
    # 프롬프트 포맷팅
    prompt = template.format(
        schema=schema,
        examples=examples,
        query_text=question
    )
    
    print("[질문]", question)
    print("\nLLM에 전달하는 프롬프트:")
    print("-" * 80)
    print(prompt)
    print("-" * 80)
    
    # LLM 호출
    llm_result = llm.invoke(prompt)
    llm_response = llm_result.content
    
    print("\n[LLM 응답]")
    print("-" * 80)
    print(llm_response)
    print("-" * 80)
    
    # Cypher 쿼리 추출
    cypher_query = extract_cypher(llm_response)
    
    print("\n[추출된 Cypher 쿼리]")
    print("-" * 80)
    print(cypher_query)
    print("-" * 80)

    return {
        'question': question,
        'prompt': prompt,
        'llm_response': llm_response,
        'cypher_query': cypher_query
    }

# 예제 1: 영화 관련 질문

In [11]:
# 예시: 영화 데이터베이스 스키마
movie_schema = """
Node properties:
- Movie {title: STRING, released: INTEGER, tagline: STRING}
- Person {name: STRING, born: INTEGER}
- Genre {name: STRING}

Relationship properties:
- ACTED_IN {roles: LIST}
- DIRECTED {}
- PRODUCED {}
- WROTE {}
- REVIEWED {rating: INTEGER, summary: STRING}
- IN_GENRE {}

The relationships:
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:PRODUCED]->(:Movie)
(:Person)-[:WROTE]->(:Movie)
(:Person)-[:REVIEWED]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)
"""

print(movie_schema)


Node properties:
- Movie {title: STRING, released: INTEGER, tagline: STRING}
- Person {name: STRING, born: INTEGER}
- Genre {name: STRING}

Relationship properties:
- ACTED_IN {roles: LIST}
- DIRECTED {}
- PRODUCED {}
- WROTE {}
- REVIEWED {rating: INTEGER, summary: STRING}
- IN_GENRE {}

The relationships:
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:PRODUCED]->(:Movie)
(:Person)-[:WROTE]->(:Movie)
(:Person)-[:REVIEWED]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)



In [12]:
question1 = "Tom Hanks가 출연한 영화를 모두 찾아줘"

result1 = text_to_cypher(
    question=question1,
    schema=movie_schema
)

[질문] Tom Hanks가 출연한 영화를 모두 찾아줘

LLM에 전달하는 프롬프트:
--------------------------------------------------------------------------------

Task: Generate a Cypher statement for querying a Neo4j graph database from a user input.

Schema:

Node properties:
- Movie {title: STRING, released: INTEGER, tagline: STRING}
- Person {name: STRING, born: INTEGER}
- Genre {name: STRING}

Relationship properties:
- ACTED_IN {roles: LIST}
- DIRECTED {}
- PRODUCED {}
- WROTE {}
- REVIEWED {rating: INTEGER, summary: STRING}
- IN_GENRE {}

The relationships:
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:PRODUCED]->(:Movie)
(:Person)-[:WROTE]->(:Movie)
(:Person)-[:REVIEWED]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)


Examples (optional):


Input:
Tom Hanks가 출연한 영화를 모두 찾아줘

Do not use any properties or relationships not included in the schema.
Do not include triple backticks ``` or any additional text except the generated Cypher statement in your response.

Cypher query:

-------------

LLMGenerationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************FYcA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}